# Librerias

In [1]:
import pandas as pd
import numpy as np
import requests
import json
import warnings
import os
import re
import gc
import kagglehub
import polars as pl
from itables import show
from tqdm import tqdm
from pathlib import Path
from scipy.spatial.distance import cdist

# Configuraciones

In [2]:
# --- 1. CONFIGURACIÓN Y CONSTANTES ---
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")

BASE_PATH = r"C:\Users\ravin\Desktop\Proyecto"
EXCEL_PATH = os.path.join(BASE_PATH, "archivos_anteriores", "Resum per mes des de 2009.xlsx")
JSON_PATH = os.path.join(BASE_PATH, "archivos_anteriores", "informacion_estaciones_bicing.json")
OUTPUT_PATH = os.path.join(BASE_PATH, "datos_procesados")
os.makedirs(OUTPUT_PATH, exist_ok=True)
INTERMEDIOS_PATH = os.path.join(OUTPUT_PATH, "intermedios")
os.makedirs(INTERMEDIOS_PATH, exist_ok=True)
os.environ["KAGGLEHUB_CACHE"] = BASE_PATH

TOKEN = "4d298cfce32e59d76e4e2f38c0a05fcd021cc5d5411a573e3c680cc51c7b5fd1"
URL_BICING = "https://opendata-ajuntament.barcelona.cat/data/dataset/bd2462df-6e1e-4e37-8205-a4b8e7313b84/resource/f60e9291-5aaa-417d-9b91-612a9de800aa/download"
URL_ELEVACION = "https://api.open-elevation.com/api/v1/lookup"
URL_KAGGLE = "edomingo/bicing-stations-dataset-bcn-bike-sharing"

MESES_ES = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
            'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
MESES_CAT = ['Gener', 'Febrer', 'Març', 'Abril', 'Maig', 'Juny',
             'Juliol', 'Agost', 'Setembre', 'Octubre', 'Novembre', 'Desembre']
CAT_A_ES = dict(zip(MESES_CAT, MESES_ES))
MES_NUM_MAP = {mes: i+1 for i, mes in enumerate(MESES_ES)}

UMBRAL_CAMION = 6 # esto puedo eliminarlo, lo he usado para detectar outliers en el número de camiones que pasan por las estaciones



# funciones

## FUNCIONES AUXILIARES

In [3]:
# --- 2. FUNCIONES AUXILIARES ---
def haversine(u, v):
    r = 6371000
    lat1, lon1 = u[0], u[1]
    lat2, lon2 = v[0], v[1]
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * r * np.arcsin(np.sqrt(a))


## PROCESAMIENTO DE EXCEL (DATOS HISTÓRICOS)

In [4]:
# --- 3. PROCESAMIENTO DE EXCEL (DATOS HISTÓRICOS) ---
def extraer_datos_excel():
    """Procesa las hojas de Usos Totales y Abonados."""
    # Valor manual diciembre 2025 (Celda M56)
    val_m56 = pd.read_excel(EXCEL_PATH, sheet_name='Usos mes', skiprows=55, nrows=1, usecols=[12], header=None).iloc[0,0]
    
    # Usos Mensuales Totales
    df_usos = pd.read_excel(EXCEL_PATH, sheet_name='Usos mes', header=3, nrows=18)
    df_usos = df_usos.loc[:, ~df_usos.columns.str.contains('^Unnamed')].copy()
    df_usos.columns = df_usos.columns.str.strip()
    df_usos.at[16, 'Desembre'] = val_m56
    df_usos_l = df_usos.melt(id_vars=['Any'], value_vars=MESES_CAT, var_name='mes', value_name='usos')
    df_usos_l['mes'] = df_usos_l['mes'].map(CAT_A_ES)
    df_usos_l['mes_num'] = df_usos_l['mes'].map(MES_NUM_MAP)
    df_usos_l = df_usos_l.rename(columns={'Any': 'año'})
    df_usos_l['usos'] = df_usos_l['usos'].astype(int)

    # 1. Cargar Abonados
    df_abon = pd.read_excel(EXCEL_PATH, sheet_name='Abonats', skiprows=3, nrows=17).iloc[:, 0:13]
    df_abon.columns = df_abon.columns.str.strip()
    df_abon = df_abon.rename(columns={'Any': 'año'})

    # 2. Corregir valores específicos de 2023 ANTES de transformar el dataframe
    # Usamos los nombres de columnas en Catalán porque aún no hemos hecho el melt
    correcciones_2023 = {
        'Maig': 140951,
        'Juny': 142198,
        'Juliol': 143815,
        'Agost': 144653
    }
    
    mask_2023 = df_abon['año'] == 2023
    for mes_cat, valor in correcciones_2023.items():
        df_abon.loc[mask_2023, mes_cat] = valor

    # 3. Asignar valor manual Diciembre 2025
    df_abon.loc[df_abon['año'] == 2025, 'Desembre'] = 166457

    # 4. Transformar a formato largo (melt)
    df_abon_l = df_abon.melt(id_vars=['año'], value_vars=MESES_CAT, var_name='mes', value_name='abonados')

    # 5. Limpieza numérica robusta
    # Convertimos a numérico (si hay strings con puntos los limpiamos, si son números los deja igual)
    def limpiar_numero(x):
        if isinstance(x, str):
            return x.replace('.', '').strip()
        return x

    df_abon_l['abonados'] = df_abon_l['abonados'].apply(limpiar_numero)
    df_abon_l['abonados'] = pd.to_numeric(df_abon_l['abonados'], errors='coerce').fillna(0).astype(int)

    # 6. Mapeos de nombres
    df_abon_l['mes'] = df_abon_l['mes'].map(CAT_A_ES)
    df_abon_l['mes_num'] = df_abon_l['mes'].map(MES_NUM_MAP)

    return df_usos_l, df_abon_l

def extraer_tipo_e_inventario():
    """Procesa el bloque de Usos por Tipo e Inventario de Bicicletas."""
    # Usos por Tipo (Mecánica vs Eléctrica)
    df_raw = pd.read_excel(EXCEL_PATH, sheet_name='Usos mes', skiprows=24, nrows=39, usecols="A:N", header=None)
    df_raw.columns = ['año_tipo'] + MESES_ES + ['total']
    df_filt = df_raw.dropna(subset=['año_tipo']).copy()
    df_filt = df_filt[~df_filt['año_tipo'].astype(str).str.contains('Dif|%|informacion|COMET', case=False)]
    df_filt[['año', 'tipo']] = df_filt['año_tipo'].str.extract(r'(\d{4})\s+(.*)')
    df_tipo_l = df_filt.melt(id_vars=['año', 'tipo'], value_vars=MESES_ES, var_name='mes', value_name='usos')
    df_tipo_l['mes_num'] = df_tipo_l['mes'].map(MES_NUM_MAP)
    df_tipo_l['usos'] = pd.to_numeric(df_tipo_l['usos'], errors='coerce').fillna(0).astype(int)

    # Inventario anual de bicicletas
    df_inv_raw = pd.read_excel(EXCEL_PATH, sheet_name='Usos mes', skiprows=24, nrows=39, usecols="A, P:S", header=None)
    df_inv_raw.columns = ['año_tipo', 'media_mec', 'media_elec', 'final_mec', 'final_elec']
    df_inv = df_inv_raw[df_inv_raw['año_tipo'].str.contains('Mecanica', case=False, na=False)].copy()
    df_inv['año'] = df_inv['año_tipo'].str.extract(r'(\d{4})')
    df_inv = df_inv[['año', 'media_mec', 'media_elec', 'final_mec', 'final_elec']]
    
    return df_tipo_l, df_inv


## PROCESAMIENTO DE ESTACIONES (API)

In [5]:

# --- 4. PROCESAMIENTO DE ESTACIONES (API) ---
def extraer_estaciones():
    """Descarga datos de Bicing, completa altitudes y calcula distancias."""
    resp = requests.get(URL_BICING, headers={"Authorization": TOKEN})
    data = resp.json()
    with open(JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)
        
    df = pd.DataFrame(data['data']['stations'])
    
    # Completar altitudes faltantes
    mask_nan = df['altitude'].isna()
    if mask_nan.any():
        print(f"Completando {mask_nan.sum()} altitudes...")
        locs = [{"latitude": r['lat'], "longitude": r['lon']} for _, r in df[mask_nan].iterrows()]
        try:
            res_alt_resp = requests.post(URL_ELEVACION, json={"locations": locs}, timeout=15)
            if res_alt_resp.status_code == 200:
                res_alt_data = res_alt_resp.json()
                df.loc[mask_nan, 'altitude'] = [r['elevation'] for r in res_alt_data['results']]
        except Exception as e:
            print(f"Aviso: No se pudo conectar con la API de elevación ({e}).")

    # Calcular distancia al vecino más cercano
    df['lat'] = pd.to_numeric(df['lat'])
    df['lon'] = pd.to_numeric(df['lon'])
    coords_rad = np.radians(df[['lat', 'lon']].values)
    dist_matrix = cdist(coords_rad, coords_rad, metric=haversine)
    np.fill_diagonal(dist_matrix, np.inf)
    df['nearby_distance_real'] = np.min(dist_matrix, axis=1).round(2)
    
    cols = ['station_id', 'name', 'lat', 'lon', 'altitude', 'address',
            'cross_street', 'post_code', 'capacity', 'is_charging_station',
            'short_name', 'nearby_distance_real']
    df['post_code'] = df['post_code'].fillna('').astype(str)
    
    return df[cols]


## PROCESAMIENTO DE KAGGLE (DATOS MASIVOS)

In [ ]:

# --- 5. PROCESAMIENTO DE KAGGLE (DATOS MASIVOS) ---
def descargar_datos_kaggle():
    """Descarga o localiza el dataset de Kaggle y devuelve la ruta."""
    print("Verificando dataset de Kaggle...")
    try:
        path = kagglehub.dataset_download(URL_KAGGLE)
        print(f"Dataset listo en: {path}")
        return path
    except Exception as e:
        print(f"Error al obtener datos de Kaggle: {e}")
        return None

# Definir columnas finales en español (las que se mantendrán)
COLUMNAS_FINALES_ES = [
    'id_estacion', 'bicis_disponibles', 'mecanicas', 'electricas',
    'anclajes_libres', 'ultimo_reporte', 'estado', 'es_cargador',
    'instalada', 'permitir_alquiler', 'permitir_devolucion',
    'ultima_actualizacion', 'año', 'mes', 'mes_num'
]

def normalizar_mensual(df, año, mes_nombre, mes_num):
    """
    Normaliza un DataFrame mensual de Kaggle:
    - Renombra columnas al español según mapeo
    - Añade año, mes, mes_num
    - Mantiene NaN en columnas numéricas (no se rellenan)
    - Columnas booleanas se mantienen como booleanas
    - Devuelve las columnas finales
    """
    rename_map = {
        'station_id': 'id_estacion',
        'num_bikes_available': 'bicis_disponibles',
        'num_bikes_available_types.mechanical': 'mecanicas_disponibles',
        'num_bikes_available_types.ebike': 'electricas_disponibles',
        'num_docks_available': 'anclajes_libres',
        'last_reported': 'ultimo_reporte',
        'status': 'estado',
        'is_charging_station': 'estacion_cargador',
        'last_updated': 'ultima_actualizacion'
    }
    
    df = df.rename(columns=rename_map)
    
    df['año'] = año
    df['mes'] = mes_nombre
    df['mes_num'] = mes_num
    
    # Convertir tipos: numéricas pueden tener NaN, no convertir a int
    df['id_estacion'] = pd.to_numeric(df['id_estacion'], errors='coerce')
    df['bicis_disponibles'] = pd.to_numeric(df['bicis_disponibles'], errors='coerce')
    df['mecanicas_disponibles'] = pd.to_numeric(df['mecanicas_disponibles'], errors='coerce')
    df['electricas_disponibles'] = pd.to_numeric(df['electricas_disponibles'], errors='coerce')
    df['anclajes_libres'] = pd.to_numeric(df['anclajes_libres'], errors='coerce')
    
    # Timestamps
    df['ultimo_reporte'] = pd.to_datetime(df['ultimo_reporte'], unit='s', errors='coerce')
    df['ultima_actualizacion'] = pd.to_datetime(df['ultima_actualizacion'], unit='s', errors='coerce')
    
    # Columnas booleanas: convertir a bool (True/False), manteniendo NaN? mejor convertir a boolean con NA
    # Para pandas, boolean con NA se puede usar 'boolean' dtype
    bool_cols = ['estacion_cargador', 'is_installed', 'is_renting', 'is_returning']
    for col in bool_cols:
        if col in df.columns:
            # Convertir a booleano, tratando 1 como True, 0 como False, otros como NA
            df[col] = df[col].map({1: True, 0: False, True: True, False: False}).astype('boolean')
        else:
            df[col] = pd.NA  # si no existe, poner NA
    
    # 'estado' es string, puede tener NaN
    if 'estado' not in df.columns:
        df['estado'] = pd.NA
    
    # Definir orden final
    columnas_finales = [
        'id_estacion', 'bicis_disponibles', 'mecanicas_disponibles', 'electricas_disponibles',
        'anclajes_libres', 'ultimo_reporte', 'estado', 'estacion_cargador',
        'is_installed', 'is_renting', 'is_returning', 'ultima_actualizacion',
        'año', 'mes', 'mes_num'
    ]
    
    return df[columnas_finales]

def procesar_datasets_kaggle(path_kaggle):
    """
    Recorre todos los archivos mensuales, los normaliza y guarda un Parquet por año.
    Retorna una lista con las rutas de los archivos Parquet anuales.
    """
    if not path_kaggle:
        return []
    
    patron = r"(\d{4})_(\d{2})_STATIONS\.csv"
    archivos_por_año = {}  # año -> lista de (ruta, mes_num)

    print("Buscando archivos mensuales en el dataset de Kaggle...")
    for raiz, dirs, archivos in os.walk(path_kaggle):
        for nombre_archivo in archivos:
            match = re.search(patron, nombre_archivo)
            if match:
                año = match.group(1)
                mes_num = int(match.group(2))
                ruta_completa = os.path.join(raiz, nombre_archivo)
                archivos_por_año.setdefault(año, []).append((ruta_completa, mes_num))
    
    rutas_parquet = []
    for año, lista_meses in archivos_por_año.items():
        print(f"Procesando año {año}...")
        dataframes_mensuales = []
        for ruta_mes, mes_num in sorted(lista_meses, key=lambda x: x[1]):
            mes_nombre = MESES_ES[mes_num - 1]
            print(f"  Leyendo {os.path.basename(ruta_mes)}...")
            try:
                # Intentar leer el CSV
                df_mes = pd.read_csv(ruta_mes, low_memory=False)
                # Verificar si está vacío (sin filas)
                if df_mes.empty:
                    print(f"  Archivo {os.path.basename(ruta_mes)} está vacío, se omite.")
                    continue
            except Exception as e:
                print(f"  Error al leer {os.path.basename(ruta_mes)}: {e}, se omite.")
                continue

            df_mes_norm = normalizar_mensual(df_mes, año, mes_nombre, mes_num)
            dataframes_mensuales.append(df_mes_norm)
            del df_mes
            gc.collect()
        
        if not dataframes_mensuales:
            print(f"  No hay datos para el año {año}, se omite.")
            continue

        # Concatenar todos los meses del año
        print(f"  Concatenando meses de {año}...")
        df_anual = pd.concat(dataframes_mensuales, ignore_index=True)
        ruta_parquet = os.path.join(OUTPUT_PATH, f"bicing_{año}.parquet")
        df_anual.to_parquet(ruta_parquet, engine='fastparquet', compression='snappy')
        print(f"  Guardado: {ruta_parquet} ({len(df_anual):,} filas)")
        rutas_parquet.append(ruta_parquet)
        
        del dataframes_mensuales, df_anual
        gc.collect()
    
    return rutas_parquet


# EJECUCIÓN PRINCIPAL

In [ ]:

# --- 6. EJECUCIÓN PRINCIPAL ---
if __name__ == "__main__":
    try:
        # 1. Procesar Excel
        print("Procesando datos históricos de Excel...")
        df_usos, df_abonados = extraer_datos_excel()
        df_tipo, df_inventario = extraer_tipo_e_inventario()
        
        # Guardar DataFrames pequeños en Parquet con fastparquet
        df_usos.to_parquet(os.path.join(INTERMEDIOS_PATH, "usos_mensuales.parquet"), engine='fastparquet')
        df_abonados.to_parquet(os.path.join(INTERMEDIOS_PATH, "abonados_mensuales.parquet"), engine='fastparquet')
        df_tipo.to_parquet(os.path.join(INTERMEDIOS_PATH, "usos_tipo.parquet"), engine='fastparquet')
        df_inventario.to_parquet(os.path.join(INTERMEDIOS_PATH, "inventario.parquet"), engine='fastparquet')
        print("Datos de Excel guardados en Parquet.")
        
        # 2. Procesar estaciones (API)
        print("Procesando estaciones desde API...")
        df_estaciones = extraer_estaciones()
        # Traducir columnas de estaciones (opcional, para mantener consistencia)
        df_estaciones = df_estaciones.rename(columns={
            'station_id': 'id_estacion',
            'name': 'nombre',
            'lat': 'latitud',
            'lon': 'longitud',
            'altitude': 'altitud',
            'address': 'direccion',
            'cross_street': 'calle_cruce',
            'post_code': 'codigo_postal',
            'capacity': 'capacidad',
            'is_charging_station': 'es_cargador',
            'short_name': 'nombre_corto',
            'nearby_distance_real': 'distancia_estacion_cercana'
        })
        df_estaciones.to_parquet(os.path.join(OUTPUT_PATH, "estaciones.parquet"), engine='fastparquet')
        print("Estaciones guardadas en Parquet.")
        
        # 3. Descargar y procesar Kaggle
        path_kaggle = descargar_datos_kaggle()
        if path_kaggle:
            rutas_anuales = procesar_datasets_kaggle(path_kaggle)
            print("Archivos anuales de Kaggle generados:")
            for r in rutas_anuales:
                print(f"  - {r}")
            
            # Opcional: unificar todos los años en un solo DataFrame
            # (solo si la memoria lo permite, unos 25-30 GB)
            unificar = False  # Cambiar a True si se desea y se tiene RAM suficiente
            if unificar and rutas_anuales:
                print("Unificando todos los años en un solo DataFrame...")
                lista_dfs = []
                for ruta in rutas_anuales:
                    df_anual = pd.read_parquet(ruta)
                    lista_dfs.append(df_anual)
                    del df_anual
                    gc.collect()
                df_total = pd.concat(lista_dfs, ignore_index=True)
                ruta_total = os.path.join(OUTPUT_PATH, "bicing_unificado.parquet")
                df_total.to_parquet(ruta_total, engine='pyarrow', compression='snappy')
                print(f"Archivo unificado guardado: {ruta_total} ({len(df_total):,} filas)")
                del lista_dfs, df_total
                gc.collect()
        else:
            print("No se pudo obtener el dataset de Kaggle.")
        
        print("\n¡Proceso completado con éxito!")
        
    except Exception as e:
        print(f"\nError general en la ejecución: {e}")

Procesando datos históricos de Excel...
Datos de Excel guardados en Parquet.
Procesando estaciones desde API...
Completando 23 altitudes...
Estaciones guardadas en Parquet.

¡Proceso completado con éxito!


### Correccion errores df abonados año 2023 meses mayo junio julio agosto

## DataFrames

In [9]:
show(df_usos)
show(df_abonados)
show(df_tipo)
show(df_inventario)
show(df_estaciones)

Loading ITables v2.7.1 from the internet... (need help?)


Loading ITables v2.7.1 from the internet... (need help?)


Loading ITables v2.7.1 from the internet... (need help?)


Loading ITables v2.7.1 from the internet... (need help?)


Loading ITables v2.7.1 from the internet... (need help?)


# Correcciones y Codigo extra

## Concatenacion archivos parquet

## Creacion del archivo concatenado que incluye 2024

In [ ]:
# 1. Cargar archivos
print("Cargando archivos...")
df_concat = pl.read_parquet(os.path.join(BASE_PATH, "archivos_anteriores", "bicing_concatenado.parquet"))
df_2024 = pl.read_parquet(os.path.join(INTERMEDIOS_PATH, "bicing_2024.parquet"))

# 2. Seleccionar las mismas columnas
columnas = ['id_estacion', 'bicis_disponibles', 'mecanicas_disponibles',
            'electricas_disponibles', 'anclajes_libres', 'estado',
            'ultima_actualizacion', 'año', 'mes', 'mes_num']
df_2024 = df_2024.select(columnas)

# 3. UNIFICAR TIPOS
columnas_numericas = ['id_estacion', 'bicis_disponibles', 'mecanicas_disponibles',
                      'electricas_disponibles', 'anclajes_libres']

for col in columnas_numericas:
    df_concat = df_concat.with_columns(pl.col(col).cast(pl.Float64))
    df_2024 = df_2024.with_columns(pl.col(col).cast(pl.Float64))

# 4. Concatenar
print("Concatenando...")
df_concat_completo = pl.concat([df_concat, df_2024])

# 5. Guardar SOBREESCRIBIENDO
print("Guardando archivo actualizado...")
df_concat_completo.write_parquet(os.path.join(BASE_PATH, "archivos_anteriores", "bicing_concatenado.parquet"))

# 6. Liberar memoria
del df_concat, df_2024, df_concat_completo

print("Archivo actualizado con 2024.")

In [3]:
df_concat = pl.read_parquet(os.path.join(BASE_PATH, "archivos_anteriores", "bicing_concatenado.parquet"))

In [14]:
# Incluido 2024
df_concat.shape

(245763007, 10)

In [16]:
df_concat.head(1)

id_estacion,bicis_disponibles,mecanicas_disponibles,electricas_disponibles,anclajes_libres,estado,ultima_actualizacion,año,mes,mes_num
f64,f64,f64,f64,f64,str,datetime[ns],str,str,i64
1.0,16.0,16.0,0.0,14.0,"""IN_SERVICE""",2019-03-28 17:58:43,"""2019""","""Marzo""",3


# Limpieza

## Paso 1: Verificación inicial
Confirmar dimensiones (filas, columnas)

Revisar tipos de datos de todas las columnas

Identificar valores nulos por columna


### Estructura df_concat (hasta 2024)

In [23]:
# DATAFRAME FINAL CON 2024
# Paso 1: Verificación inicial
print("=== VERIFICACIÓN INICIAL ===\n")

# 1.1 Dimensiones
print(f"Dimensiones: {df_concat.shape[0]:,} filas × {df_concat.shape[1]} columnas\n")

# 1.2 Estructura y tipos
print("Estructura y tipos de datos:")
print(df_concat.schema)

# 1.3 Valores nulos por columna
print("\nValores nulos por columna:")
print(df_concat.null_count())

# 1.4 Vista previa de los datos
print("\nVista previa (primeras 5 filas):")
print(df_concat.head(5))

# 1.5 Estadísticas básicas de columnas numéricas
print("\nEstadísticas básicas (numéricas):")
print(df_concat.select(pl.all().exclude(['estado', 'mes', 'ultima_actualizacion'])).describe())

# 1.6 Valores únicos en columna 'estado'
print("\nValores únicos en 'estado':")
print(df_concat['estado'].unique().to_list())

=== VERIFICACIÓN INICIAL ===

Dimensiones: 245,763,007 filas × 17 columnas

Estructura y tipos de datos:
Schema({'id_estacion': Float64, 'bicis_disponibles': Float64, 'mecanicas_disponibles': Float64, 'electricas_disponibles': Float64, 'anclajes_libres': Float64, 'estado': String, 'ultima_actualizacion': Datetime(time_unit='ns', time_zone=None), 'año': String, 'mes': String, 'mes_num': Int64, 'capacidad_calculada': Float64, 'tasa_ocupacion': Float64, 'estacion_vacia': Boolean, 'estacion_llena': Boolean, 'casi_vacia': Boolean, 'casi_llena': Boolean, 'timestamp_seconds': Int64})

Valores nulos por columna:
shape: (1, 17)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id_estaci ┆ bicis_dis ┆ mecanicas ┆ electrica ┆ … ┆ estacion_ ┆ casi_vaci ┆ casi_llen ┆ timestam │
│ on        ┆ ponibles  ┆ _disponib ┆ s_disponi ┆   ┆ llena     ┆ a         ┆ a         ┆ p_second │
│ ---       ┆ ---       ┆ les       ┆ bles      ┆   ┆ ---       ┆ --- 

## Paso 2: Conversión de tipos y limpieza de columnas numéricas
id_estacion → asegurar que es entero (ya debería)

bicis_disponibles, mecanicas_disponibles, electricas_disponibles, anclajes_libres → convertir a entero (si hay nulos, luego vemos que hacer con ellos)

Verificar que no haya valores negativos (incoherentes)

In [4]:
# Convertir columnas numéricas a enteros (si es posible, manteniendo NaN)
columnas_int = ['id_estacion', 'bicis_disponibles', 'mecanicas_disponibles', 
                'electricas_disponibles', 'anclajes_libres']

for col in columnas_int:
    df_concat = df_concat.with_columns(
        pl.col(col).cast(pl.Int64)
    )

# 'año' debería ser entero (está en string por ahora)
df_concat = df_concat.with_columns(
    pl.col('año').cast(pl.Int32).alias('año')
)

In [21]:
#Compruebo tipos y nulos después de la conversión
# Estructura y tipos
print("Estructura y tipos de datos:")
print(df_concat.schema)
# Valores nulos por columna
print("\nValores nulos por columna:")
print(df_concat.null_count())

Estructura y tipos de datos:
Schema({'id_estacion': Int64, 'bicis_disponibles': Int64, 'mecanicas_disponibles': Int64, 'electricas_disponibles': Int64, 'anclajes_libres': Int64, 'estado': String, 'ultima_actualizacion': Datetime(time_unit='ns', time_zone=None), 'año': Int32, 'mes': String, 'mes_num': Int64})

Valores nulos por columna:
shape: (1, 10)
┌─────────────┬──────────────┬──────────────┬──────────────┬───┬─────────────┬─────┬─────┬─────────┐
│ id_estacion ┆ bicis_dispon ┆ mecanicas_di ┆ electricas_d ┆ … ┆ ultima_actu ┆ año ┆ mes ┆ mes_num │
│ ---         ┆ ibles        ┆ sponibles    ┆ isponibles   ┆   ┆ alizacion   ┆ --- ┆ --- ┆ ---     │
│ u32         ┆ ---          ┆ ---          ┆ ---          ┆   ┆ ---         ┆ u32 ┆ u32 ┆ u32     │
│             ┆ u32          ┆ u32          ┆ u32          ┆   ┆ u32         ┆     ┆     ┆         │
╞═════════════╪══════════════╪══════════════╪══════════════╪═══╪═════════════╪═════╪═════╪═════════╡
│ 3776        ┆ 3776         ┆ 3776      

In [22]:
# Visualizacion de Filas con algún valor negativo en las columnas de bicicletas
negativos = df_concat.filter(
    (pl.col('bicis_disponibles') < 0) |
    (pl.col('mecanicas_disponibles') < 0) |
    (pl.col('electricas_disponibles') < 0)
)

print(f"Filas con valores negativos: {negativos.shape[0]}")
negativos.sort('bicis_disponibles', descending=False).head(10)

Filas con valores negativos: 3565


id_estacion,bicis_disponibles,mecanicas_disponibles,electricas_disponibles,anclajes_libres,estado,ultima_actualizacion,año,mes,mes_num
i64,i64,i64,i64,i64,str,datetime[ns],i32,str,i64
239,-2,-2,0,26,"""IN_SERVICE""",2019-07-10 06:09:52,2019,"""Julio""",7
103,-2,-2,0,19,"""IN_SERVICE""",2019-07-18 06:45:07,2019,"""Julio""",7
206,-2,0,-2,30,"""IN_SERVICE""",2024-03-05 17:40:01,2024,"""Marzo""",3
207,-2,0,-2,24,"""IN_SERVICE""",2024-03-07 17:10:03,2024,"""Marzo""",3
207,-2,0,-2,23,"""IN_SERVICE""",2024-03-12 17:35:04,2024,"""Marzo""",3
319,-2,0,-2,31,"""IN_SERVICE""",2024-03-18 17:25:03,2024,"""Marzo""",3
121,-1,-1,0,15,"""IN_SERVICE""",2019-05-08 07:19:59,2019,"""Mayo""",5
73,-1,-1,0,27,"""IN_SERVICE""",2019-05-13 17:30:06,2019,"""Mayo""",5
350,-1,-1,0,31,"""IN_SERVICE""",2019-05-13 17:30:06,2019,"""Mayo""",5


## Paso 3: Limpieza de la columna estado
Verificar valores únicos

Decidir si filtrar solo "IN_SERVICE" o mantener todos los estados (los mantendremos)

Si se mantienen, estandarizar nombres (mayúsculas, espacios. Están estandarizados)

In [23]:
# 1. Ver valores únicos en 'estado'
print("Valores únicos en 'estado':")
print(df_concat['estado'].unique().to_list())

# 2. Contar frecuencia de cada estado
print("\nFrecuencia de cada estado:")
print(df_concat.group_by('estado').len().sort('len', descending=True))

Valores únicos en 'estado':
['MAINTENANCE', 'END_OF_LIFE', 'NOT_IN_SERVICE', None, 'PLANNED', 'IN_SERVICE']

Frecuencia de cada estado:
shape: (6, 2)
┌────────────────┬───────────┐
│ estado         ┆ len       │
│ ---            ┆ ---       │
│ str            ┆ u32       │
╞════════════════╪═══════════╡
│ IN_SERVICE     ┆ 241706783 │
│ MAINTENANCE    ┆ 3266511   │
│ PLANNED        ┆ 477141    │
│ NOT_IN_SERVICE ┆ 308788    │
│ null           ┆ 3776      │
│ END_OF_LIFE    ┆ 8         │
└────────────────┴───────────┘


## Paso 4: Normalización de fechas
Verificar que ultima_actualizacion es datetime (ya lo es)

Extraer componentes útiles si no existen: hora, dia_semana, mes, año (ya tenemos año y mes)

Crear columna fecha (solo YYYY-MM-DD) para agrupaciones diarias

In [8]:
# Normalización de fechas

# 1. Verificar tipo (ya es datetime)
print("Tipo de ultima_actualizacion:", df_concat['ultima_actualizacion'].dtype)

# 2. Extraer componentes útiles
df_concat = df_concat.with_columns([
    pl.col('ultima_actualizacion').dt.hour().alias('hora'),
    pl.col('ultima_actualizacion').dt.weekday().alias('dia_semana'),  # 1=lunes, 7=domingo
    pl.col('ultima_actualizacion').dt.day().alias('dia_mes')
])

# 3. Crear columna fecha (solo YYYY-MM-DD)
df_concat = df_concat.with_columns(
    pl.col('ultima_actualizacion').dt.date().alias('fecha')
)

# 4. Verificar nuevas columnas
print("\nNuevas columnas añadidas:")
print(df_concat.select(['hora', 'dia_semana', 'dia_mes', 'fecha']).head())

Tipo de ultima_actualizacion: Datetime(time_unit='ns', time_zone=None)

Nuevas columnas añadidas:
shape: (5, 4)
┌──────┬────────────┬─────────┬────────────┐
│ hora ┆ dia_semana ┆ dia_mes ┆ fecha      │
│ ---  ┆ ---        ┆ ---     ┆ ---        │
│ i8   ┆ i8         ┆ i8      ┆ date       │
╞══════╪════════════╪═════════╪════════════╡
│ 17   ┆ 4          ┆ 28      ┆ 2019-03-28 │
│ 17   ┆ 4          ┆ 28      ┆ 2019-03-28 │
│ 17   ┆ 4          ┆ 28      ┆ 2019-03-28 │
│ 17   ┆ 4          ┆ 28      ┆ 2019-03-28 │
│ 17   ┆ 4          ┆ 28      ┆ 2019-03-28 │
└──────┴────────────┴─────────┴────────────┘


## Paso 5: Validación de consistencia
Ver máximos de bicis y anclajes

Verificar que bicis_disponibles = mecanicas_disponibles + electricas_disponibles

Verificar que bicis_disponibles + anclajes_libres <= capacidad de la estación (con df_estaciones)

Si hay discrepancias, decidir cómo manejarlas

### 5.1 Ver máximos de bicis y anclajes

In [25]:
# Ver máximos
print("Máximos observados:")
print(f"bicis_disponibles: {df_concat['bicis_disponibles'].max()}")
print(f"anclajes_libres: {df_concat['anclajes_libres'].max()}")

Máximos observados:
bicis_disponibles: 198
anclajes_libres: 99


In [26]:
# 1. Ver la estación con 198 bicis disponibles
max_bicis = df_concat.filter(pl.col('bicis_disponibles') == 198)
print("Estación con 198 bicis disponibles:")
max_bicis

Estación con 198 bicis disponibles:


id_estacion,bicis_disponibles,mecanicas_disponibles,electricas_disponibles,anclajes_libres,estado,ultima_actualizacion,año,mes,mes_num,hora,dia_semana,dia_mes,fecha
i64,i64,i64,i64,i64,str,datetime[ns],i32,str,i64,i8,i8,i8,date
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 07:15:26,2020,"""Agosto""",8,7,3,12,2020-08-12
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 07:20:28,2020,"""Agosto""",8,7,3,12,2020-08-12
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 07:25:08,2020,"""Agosto""",8,7,3,12,2020-08-12
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 08:15:15,2020,"""Agosto""",8,8,3,12,2020-08-12
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 08:20:08,2020,"""Agosto""",8,8,3,12,2020-08-12
…,…,…,…,…,…,…,…,…,…,…,…,…,…
535,198,99,99,99,"""IN_SERVICE""",2024-03-18 09:21:04,2024,"""Marzo""",3,9,1,18,2024-03-18
535,198,99,99,99,"""IN_SERVICE""",2024-03-18 09:25:01,2024,"""Marzo""",3,9,1,18,2024-03-18
535,198,99,99,99,"""IN_SERVICE""",2024-03-18 09:30:03,2024,"""Marzo""",3,9,1,18,2024-03-18


In [27]:
# 2. Ver todos los registros de esa estación
estacion_id = max_bicis['id_estacion'][0]
df_estacion = df_concat.filter(pl.col('id_estacion') == estacion_id)
print(f"\nTodos los registros de estación {estacion_id}:")
print(df_estacion.select(['ultima_actualizacion', 'bicis_disponibles', 
                          'anclajes_libres', 'estado']).sort('ultima_actualizacion'))


Todos los registros de estación 529:
shape: (190, 4)
┌──────────────────────┬───────────────────┬─────────────────┬────────────┐
│ ultima_actualizacion ┆ bicis_disponibles ┆ anclajes_libres ┆ estado     │
│ ---                  ┆ ---               ┆ ---             ┆ ---        │
│ datetime[ns]         ┆ i64               ┆ i64             ┆ str        │
╞══════════════════════╪═══════════════════╪═════════════════╪════════════╡
│ 2020-08-12 07:15:26  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-08-12 07:20:28  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-08-12 07:25:08  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-08-12 08:15:15  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-08-12 08:20:08  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ …                    ┆ …                 ┆ …               ┆ …          │
│ 2020-12-31 17:25:29  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-12-31 17:30:23  ┆ 198      

In [28]:
# 3. Ver la estación con 99 anclajes libres
max_anclajes = df_concat.filter(pl.col('anclajes_libres') == 99)
print("\nEstación con 99 anclajes libres:")
max_anclajes


Estación con 99 anclajes libres:


id_estacion,bicis_disponibles,mecanicas_disponibles,electricas_disponibles,anclajes_libres,estado,ultima_actualizacion,año,mes,mes_num,hora,dia_semana,dia_mes,fecha
i64,i64,i64,i64,i64,str,datetime[ns],i32,str,i64,i8,i8,i8,date
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 07:15:26,2020,"""Agosto""",8,7,3,12,2020-08-12
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 07:20:28,2020,"""Agosto""",8,7,3,12,2020-08-12
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 07:25:08,2020,"""Agosto""",8,7,3,12,2020-08-12
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 08:15:15,2020,"""Agosto""",8,8,3,12,2020-08-12
529,198,99,99,99,"""IN_SERVICE""",2020-08-12 08:20:08,2020,"""Agosto""",8,8,3,12,2020-08-12
…,…,…,…,…,…,…,…,…,…,…,…,…,…
535,198,99,99,99,"""IN_SERVICE""",2024-03-18 09:21:04,2024,"""Marzo""",3,9,1,18,2024-03-18
535,198,99,99,99,"""IN_SERVICE""",2024-03-18 09:25:01,2024,"""Marzo""",3,9,1,18,2024-03-18
535,198,99,99,99,"""IN_SERVICE""",2024-03-18 09:30:03,2024,"""Marzo""",3,9,1,18,2024-03-18


In [29]:
# 4. Ver todos los registros de esa estación
estacion_anclaje_id = max_anclajes['id_estacion'][0]
df_estacion_anclaje = df_concat.filter(pl.col('id_estacion') == estacion_anclaje_id)
print(f"\nTodos los registros de estación {estacion_anclaje_id}:")
print(df_estacion_anclaje.select(['ultima_actualizacion', 'bicis_disponibles', 
                                  'anclajes_libres', 'estado']).sort('ultima_actualizacion'))


Todos los registros de estación 529:
shape: (190, 4)
┌──────────────────────┬───────────────────┬─────────────────┬────────────┐
│ ultima_actualizacion ┆ bicis_disponibles ┆ anclajes_libres ┆ estado     │
│ ---                  ┆ ---               ┆ ---             ┆ ---        │
│ datetime[ns]         ┆ i64               ┆ i64             ┆ str        │
╞══════════════════════╪═══════════════════╪═════════════════╪════════════╡
│ 2020-08-12 07:15:26  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-08-12 07:20:28  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-08-12 07:25:08  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-08-12 08:15:15  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-08-12 08:20:08  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ …                    ┆ …                 ┆ …               ┆ …          │
│ 2020-12-31 17:25:29  ┆ 198               ┆ 99              ┆ IN_SERVICE │
│ 2020-12-31 17:30:23  ┆ 198      

### Marcar datos como validos (true/false)

In [16]:
# Definir rango realista con validación de capacidad combinada
df_concat = df_concat.with_columns(
    (
        (pl.col('bicis_disponibles') >= 0) & 
        (pl.col('bicis_disponibles') <= 60) &
        (pl.col('anclajes_libres') >= 0) &
        (pl.col('anclajes_libres') <= 60) &
        # NUEVO: la suma también debe ser realista
        ((pl.col('bicis_disponibles') + pl.col('anclajes_libres')) >= 1) &
        ((pl.col('bicis_disponibles') + pl.col('anclajes_libres')) <= 60)
    ).alias('dato_valido_bicis')
)

# Ver cuántos son válidos vs no válidos
print("Registros válidos:", df_concat.filter(pl.col('dato_valido_bicis') == True).shape[0])
print("Registros no válidos:", df_concat.filter(pl.col('dato_valido_bicis') == False).shape[0])

# Visualizar todos los ids de estaciones con datos no válidos
ids_invalidos = df_concat.filter(pl.col('dato_valido_bicis') == False)['id_estacion'].unique().sort()
print(f"\nIDs de estaciones con datos no válidos ({len(ids_invalidos)} estaciones):")
print(ids_invalidos.to_list())

Registros válidos: 245592591
Registros no válidos: 166640

IDs de estaciones con datos no válidos (503 estaciones):
[1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 170, 171, 173, 174, 175, 176, 177, 178, 179, 180, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 20

In [ ]:
# # Definir rango realista según pliego (capacidad máxima 60 anclajes)
# df_concat = df_concat.with_columns(
#     ((pl.col('bicis_disponibles') >= 0) & 
#      (pl.col('bicis_disponibles') <= 60) &
#      (pl.col('anclajes_libres') >= 0) &
#      (pl.col('anclajes_libres') <= 60)).alias('dato_valido_bicis')
# )

# # Ver cuántos son válidos vs no válidos
# print("Registros válidos:", df_concat.filter(pl.col('dato_valido_bicis') == True).shape[0])
# print("Registros no válidos:", df_concat.filter(pl.col('dato_valido_bicis') == False).shape[0])

# # Visualizar todos los ids de estaciones con datos no válidos
# ids_invalidos = df_concat.filter(pl.col('dato_valido_bicis') == False)['id_estacion'].unique().sort()
# print(f"\nIDs de estaciones con datos no válidos ({len(ids_invalidos)} estaciones):")
# print(ids_invalidos.to_list())

Registros válidos: 245755877
Registros no válidos: 3354

IDs de estaciones con datos no válidos (281 estaciones):
[1, 2, 6, 15, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 36, 41, 42, 43, 44, 48, 50, 51, 54, 60, 61, 62, 63, 64, 65, 66, 67, 68, 70, 71, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 106, 107, 108, 109, 110, 111, 112, 113, 116, 119, 120, 121, 122, 123, 127, 129, 133, 136, 139, 140, 142, 143, 145, 149, 150, 151, 152, 156, 161, 164, 166, 168, 170, 175, 177, 183, 185, 186, 187, 188, 189, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 211, 212, 213, 214, 215, 216, 217, 218, 220, 221, 222, 223, 224, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 239, 241, 242, 246, 250, 251, 261, 262, 265, 273, 276, 277, 278, 279, 280, 281, 282, 286, 287, 302, 303, 304, 305, 306, 312, 313, 314, 315, 318, 319, 320, 321, 322, 323, 324, 325

### 5.2 Crear columna "capacidad_calculada"

In [17]:
# Capacidad calculada
df_concat = df_concat.with_columns(
    (pl.col('bicis_disponibles') + pl.col('anclajes_libres')).alias('capacidad_calculada')
)

# Verificar — con el nuevo filtro estos valores deberían desaparecer
print("Estadísticas de capacidad_calculada:")
print(df_concat['capacidad_calculada'].describe())
# Esperado: min >= 1, max <= 60, null_count = 3776 (los nulos de estado)

# Verificar capacidad solo sobre registros válidos
print("Estadísticas de capacidad_calculada (solo registros válidos):")
print(
    df_concat
    .filter(pl.col('dato_valido_bicis') == True)
    ['capacidad_calculada']
    .describe()
)

# Contar cuántos registros válidos tienen capacidad == 0
n_cap_cero = (
    df_concat
    .filter(
        (pl.col('dato_valido_bicis') == True) &
        (pl.col('capacidad_calculada') == 0)
    )
    .height
)
print(f"\nRegistros válidos con capacidad == 0: {n_cap_cero}")

# Contar cuántos registros válidos tienen tasa_ocupacion nula
n_ocup_nula = (
    df_concat
    .filter(
        (pl.col('dato_valido_bicis') == True) &
        pl.col('tasa_ocupacion').is_null()
    )
    .height
)
print(f"Registros válidos con tasa_ocupacion nula: {n_ocup_nula}")



Estadísticas de capacidad_calculada:
shape: (9, 2)
┌────────────┬──────────────┐
│ statistic  ┆ value        │
│ ---        ┆ ---          │
│ str        ┆ f64          │
╞════════════╪══════════════╡
│ count      ┆ 2.45759231e8 │
│ null_count ┆ 3776.0       │
│ mean       ┆ 25.325967    │
│ std        ┆ 6.599509     │
│ min        ┆ -1.0         │
│ 25%        ┆ 21.0         │
│ 50%        ┆ 25.0         │
│ 75%        ┆ 27.0         │
│ max        ┆ 297.0        │
└────────────┴──────────────┘
Estadísticas de capacidad_calculada (solo registros válidos):
shape: (9, 2)
┌────────────┬──────────────┐
│ statistic  ┆ value        │
│ ---        ┆ ---          │
│ str        ┆ f64          │
╞════════════╪══════════════╡
│ count      ┆ 2.45592591e8 │
│ null_count ┆ 0.0          │
│ mean       ┆ 25.340988    │
│ std        ┆ 6.531328     │
│ min        ┆ 1.0          │
│ 25%        ┆ 21.0         │
│ 50%        ┆ 25.0         │
│ 75%        ┆ 27.0         │
│ max        ┆ 55.0         │
└──

In [ ]:
# # Crear capacidad calculada (bicis disponibles + anclajes libres)
# df_concat = df_concat.with_columns(
#     (pl.col('bicis_disponibles') + pl.col('anclajes_libres')).alias('capacidad_calculada')
# )

# # Ver estadísticas
# print("Estadísticas de capacidad_calculada:")
# print(df_concat['capacidad_calculada'].describe())

Estadísticas de capacidad_calculada:
shape: (9, 2)
┌────────────┬──────────────┐
│ statistic  ┆ value        │
│ ---        ┆ ---          │
│ str        ┆ f64          │
╞════════════╪══════════════╡
│ count      ┆ 2.45759231e8 │
│ null_count ┆ 3776.0       │
│ mean       ┆ 25.325967    │
│ std        ┆ 6.599509     │
│ min        ┆ -1.0         │
│ 25%        ┆ 21.0         │
│ 50%        ┆ 25.0         │
│ 75%        ┆ 27.0         │
│ max        ┆ 297.0        │
└────────────┴──────────────┘


In [18]:
# Reordenar columnas para mejor visualización

columnas_ordenadas = [
    'id_estacion', 
    'estado', 
    'capacidad_calculada',
    'bicis_disponibles', 
    'mecanicas_disponibles', 
    'electricas_disponibles', 
    'anclajes_libres', 
    'ultima_actualizacion', 
    'fecha', 
    'año', 
    'mes', 
    'dia_mes', 
    'dia_semana', 
    'hora', 
    'mes_num',
    'dato_valido_bicis'
]

df_concat = df_concat.select(columnas_ordenadas)

print("Columnas reordenadas:")
df_concat.head(1)

Columnas reordenadas:


id_estacion,estado,capacidad_calculada,bicis_disponibles,mecanicas_disponibles,electricas_disponibles,anclajes_libres,ultima_actualizacion,fecha,año,mes,dia_mes,dia_semana,hora,mes_num,dato_valido_bicis
i64,str,i64,i64,i64,i64,i64,datetime[ns],date,i32,str,i8,i8,i8,i8,bool
1,"""IN_SERVICE""",30,16,16,0,14,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true


### 5.3 Comparar con df_estaciones (2026)
no se como mantener la coherencia entre estaciones de 2019 y el df de 2026

## Paso 6: Eliminación de duplicados
Verificar si hay filas completamente duplicadas (mismo id_estacion y misma ultima_actualizacion)

Eliminar duplicados si existen

### Verificacion de duplicados
cada vez que corro el programa, el kernel muere

In [33]:
# # Verificar duplicados exactos (misma estación, misma fecha/hora)
# duplicados = df_concat.group_by(['id_estacion', 'ultima_actualizacion']).len()
# duplicados_totales = duplicados.filter(pl.col('len') > 1).shape[0]

# print(f"Registros duplicados (misma estación y timestamp): {duplicados_totales}")

# Eliminar duplicados (mantener el primero)
# df_concat = df_concat.unique(
#     subset=['id_estacion', 'ultima_actualizacion'], 
#     keep='first'
# )

#print(f"Dimensiones después de eliminar duplicados: {df_concat.shape[0]:,} filas × {df_concat.shape[1]} columnas")

## Paso 7: Columnas adicionales para análisis avanzado:


In [19]:
# 1. Tasa de ocupación — proteger división por cero
# Si capacidad_calculada == 0, el sensor ha fallado → producir null, no NaN
df_concat = df_concat.with_columns(
    pl.when(pl.col('capacidad_calculada') > 0)
      .then(pl.col('bicis_disponibles') / pl.col('capacidad_calculada') * 100)
      .otherwise(None)                          # null explícito, no NaN
      .alias('tasa_ocupacion')
)

# 2. Indicadores de saturación
df_concat = df_concat.with_columns([
    (pl.col('bicis_disponibles') == 0).alias('estacion_vacia'),
    (pl.col('anclajes_libres') == 0).alias('estacion_llena'),

    pl.when(pl.col('capacidad_calculada') > 0)
      .then((pl.col('bicis_disponibles') / pl.col('capacidad_calculada')) <= 0.1)
      .otherwise(None)
      .alias('casi_vacia'),

    pl.when(pl.col('capacidad_calculada') > 0)
      .then((pl.col('bicis_disponibles') / pl.col('capacidad_calculada')) >= 0.9)
      .otherwise(None)
      .alias('casi_llena')
])

# 3. Para predicción: columna de tiempo continuo (timestamp en segundos desde inicio)
min_date = df_concat['ultima_actualizacion'].min()
df_concat = df_concat.with_columns(
    (pl.col('ultima_actualizacion') - min_date).dt.total_seconds().alias('timestamp_seconds')
)

# 4. Lag features para predicción (esto se hará en el modelo, no en el dataset general)

# 5. Estacionalidad adicional
df_concat = df_concat.with_columns([
    pl.col('mes_num').cast(pl.Int8).alias('mes_num'),
    pl.col('hora').cast(pl.Int8).alias('hora'),
    pl.col('dia_semana').cast(pl.Int8).alias('dia_semana')
])

# 6. Columna para identificar fin de semana
df_concat = df_concat.with_columns(
    (pl.col('dia_semana') >= 6).alias('es_fin_semana')
)

In [ ]:
df_concat

id_estacion,estado,capacidad_calculada,bicis_disponibles,mecanicas_disponibles,electricas_disponibles,anclajes_libres,ultima_actualizacion,fecha,año,mes,dia_mes,dia_semana,hora,mes_num,dato_valido_bicis,tasa_ocupacion,estacion_vacia,estacion_llena,casi_vacia,casi_llena,timestamp_seconds,es_fin_semana
i64,str,i64,i64,i64,i64,i64,datetime[ns],date,i32,str,i8,i8,i8,i8,bool,f64,bool,bool,bool,bool,i64,bool
1,"""IN_SERVICE""",30,16,16,0,14,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true,53.333333,false,false,false,false,0,false
2,"""IN_SERVICE""",27,27,27,0,0,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true,100.0,false,true,false,true,0,false
3,"""IN_SERVICE""",27,20,20,0,7,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true,74.074074,false,false,false,false,0,false
4,"""IN_SERVICE""",19,12,12,0,7,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true,63.157895,false,false,false,false,0,false
5,"""IN_SERVICE""",39,39,39,0,0,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true,100.0,false,true,false,true,0,false
6,"""IN_SERVICE""",36,36,36,0,0,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true,100.0,false,true,false,true,0,false
7,"""IN_SERVICE""",27,26,26,0,1,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true,96.296296,false,false,false,true,0,false
8,"""IN_SERVICE""",26,26,26,0,0,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true,100.0,false,true,false,true,0,false
9,"""IN_SERVICE""",27,23,23,0,4,2019-03-28 17:58:43,2019-03-28,2019,"""Marzo""",28,4,17,3,true,85.185185,false,false,false,false,0,false


## Paso 8: Guardado del DataFrame limpio
Guardar en formato Parquet optimizado (compresión snappy o zstd) para futuros análisis

Opcional: guardar también en formato Polars (.ipc) para lectura ultrarrápida

In [20]:
# Guardar el DataFrame limpio en raw
ruta_guardado = os.path.join(OUTPUT_PATH, "raw", "bicing_limpio.parquet")
df_concat.write_parquet(ruta_guardado)

print(f"Archivo guardado: {ruta_guardado}")
print(f"Dimensiones: {df_concat.shape[0]:,} filas × {df_concat.shape[1]} columnas")
print(f"Tamaño aproximado en disco: ~{os.path.getsize(ruta_guardado) / (1024**3):.2f} GB")

Archivo guardado: C:\Users\ravin\Desktop\Proyecto\datos_procesados\raw\bicing_limpio.parquet
Dimensiones: 245,763,007 filas × 23 columnas
Tamaño aproximado en disco: ~0.76 GB


In [22]:
import polars as pl
import os

BASE_PATH = r"C:\Users\ravin\Desktop\Proyecto"
OUTPUT_PATH = os.path.join(BASE_PATH, "datos_procesados")
AGREGADOS_PATH = os.path.join(OUTPUT_PATH, "agregados")

# 1. Cargar dataset_mapa
df_mapa = pl.read_parquet(os.path.join(AGREGADOS_PATH, "dataset_mapa.parquet"))
print(f"1. dataset_mapa: {df_mapa.shape[0]} estaciones")

# 2. Cargar ranking
ranking = pl.read_parquet(os.path.join(AGREGADOS_PATH, "ranking_estaciones_completo.parquet"))
print(f"2. ranking: {ranking.shape[0]} estaciones")

# 3. Cargar info estaciones enriquecida
info_est = pl.read_parquet(os.path.join(AGREGADOS_PATH, "info_estaciones_enriquecida.parquet"))
print(f"3. info_estaciones_enriquecida: {info_est.shape[0]} estaciones")

# 4. Ver cuántas tienen coordenadas válidas
print(f"4. Con latitud no nula: {df_mapa.filter(pl.col('latitud').is_not_null()).shape[0]}")
print(f"5. Con longitud no nula: {df_mapa.filter(pl.col('longitud').is_not_null()).shape[0]}")

# 5. Ver las primeras filas
print("\nPrimeras 5 filas de df_mapa:")
print(df_mapa.head())

1. dataset_mapa: 544 estaciones
2. ranking: 505 estaciones
3. info_estaciones_enriquecida: 544 estaciones
4. Con latitud no nula: 544
5. Con longitud no nula: 544

Primeras 5 filas de df_mapa:
shape: (5, 16)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id_estaci ┆ nombre    ┆ latitud   ┆ longitud  ┆ … ┆ electrica ┆ anclajes_ ┆ rotacion_ ┆ rotacion │
│ on        ┆ ---       ┆ ---       ┆ ---       ┆   ┆ s_media   ┆ media     ┆ total_med ┆ _filtrad │
│ ---       ┆ str       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ia        ┆ a_media  │
│ i64       ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1         ┆ GRAN VIA  ┆ 41.397978 ┆ 2.1801069 ┆ … ┆ 1.507127  ┆ 26.245013 ┆ 0.92788

In [12]:
from itables import show
show(info_est)

Loading ITables v2.7.1 from the internet... (need help?)


In [29]:
from itables import show
show(df_mapa.sort("pct_vacia", descending=True))

Loading ITables v2.7.1 from the internet... (need help?)


In [11]:
from itables import show
show(ranking)


Loading ITables v2.7.1 from the internet... (need help?)


In [9]:
import polars as pl
import os

OUTPUT_PATH = r"C:\Users\ravin\Desktop\Proyecto\datos_procesados"
df_est = pl.read_parquet(os.path.join(OUTPUT_PATH, "estaciones.parquet"))

BBOX_LAT = (41.32, 41.47)
BBOX_LON = (2.05,  2.23)
PALABRAS = ['prueba', 'test', 'smartcity', 'smart city',
            'copa america', 'copa américa', 'temporal', 'demo']

excluidas = df_est.filter(
    (pl.col('latitud')  < BBOX_LAT[0]) |
    (pl.col('latitud')  > BBOX_LAT[1]) |
    (pl.col('longitud') < BBOX_LON[0]) |
    (pl.col('longitud') > BBOX_LON[1]) |
    pl.col('nombre').str.to_lowercase().str.contains('|'.join(PALABRAS))
)

print(f"Estaciones que serían excluidas: {len(excluidas)}")
print(excluidas.select(['id_estacion', 'nombre', 'latitud', 'longitud']))

Estaciones que serían excluidas: 2
shape: (2, 4)
┌─────────────┬──────────────────────────────┬───────────┬──────────┐
│ id_estacion ┆ nombre                       ┆ latitud   ┆ longitud │
│ ---         ┆ ---                          ┆ ---       ┆ ---      │
│ i64         ┆ str                          ┆ f64       ┆ f64      │
╞═════════════╪══════════════════════════════╪═══════════╪══════════╡
│ 542         ┆ Copa América Barcelona - 542 ┆ 41.374538 ┆ 2.189217 │
│ 543         ┆ Copa América Barcelona - 543 ┆ 41.38383  ┆ 2.191371 │
└─────────────┴──────────────────────────────┴───────────┴──────────┘
